# Tahap 5: Demo CBR — Pencarian Kasus Serupa

Notebook ini mendemonstrasikan pencarian kasus serupa menggunakan **Weighted Feature Similarity**.

### Komponen Similaritas
| Komponen | Bobot | Keterangan |
|----------|-------|------------|
| Kata kunci | 40% | Tumpang-tindih kata kunci (Jaccard) |
| Jenis perkara | 30% | Kecocokan kategori |
| Pengadilan | 15% | Kecocokan nama pengadilan |
| Kesamaan teks | 15% | Proporsi kata yang sama |

- **Input:** `../data/processed/case_base/case_base_*.json` (hasil Tahap 4)
- **Output:** Tampilan hasil + `../data/processed/case_base/hasil_cbr_*.csv`

In [ ]:
!pip install pandas -q
print('Instalasi selesai!')

## Muat Case Base

In [ ]:
import os
import re
import json
import glob
import pandas as pd
from collections import Counter
from datetime import datetime

CASE_BASE_DIR = '../data/processed/case_base'

json_files = sorted(glob.glob(os.path.join(CASE_BASE_DIR, 'case_base_*.json')))

if not json_files:
    print('Tidak ada file case base di ' + CASE_BASE_DIR)
    print('Jalankan notebook 04_bangun_case_base.ipynb terlebih dahulu.')
    case_base = []
else:
    print('File case base tersedia:')
    for i, f in enumerate(json_files):
        print('  [' + str(i) + '] ' + os.path.basename(f))
    try:
        idx = int(input('Pilih nomor file [default=' + str(len(json_files)-1) + ']: ').strip() or str(len(json_files)-1))
    except ValueError:
        idx = len(json_files) - 1
    with open(json_files[idx], 'r', encoding='utf-8') as f:
        case_base = json.load(f)
    print('Case base dimuat: ' + str(len(case_base)) + ' kasus dari ' + os.path.basename(json_files[idx]))

## Fungsi Similaritas CBR

In [ ]:
STOPWORDS = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 'adalah',
    'ini', 'itu', 'dalam', 'oleh', 'sebagai', 'atau', 'bahwa', 'telah',
    'tidak', 'akan', 'juga', 'serta', 'tersebut', 'dapat', 'harus',
}


def ekstrak_kata_kunci(teks, top_n=15):
    kata = re.findall(r'\b[a-zA-Z]{4,}\b', teks.lower())
    bersih = [k for k in kata if k not in STOPWORDS]
    return [k for k, _ in Counter(bersih).most_common(top_n)]


def hitung_similaritas(query, kasus_db, bobot=None):
    if bobot is None:
        bobot = {'kata_kunci': 0.40, 'jenis_perkara': 0.30, 'pengadilan': 0.15, 'teks': 0.15}

    skor = 0.0

    # 1. Kecocokan kata kunci (Jaccard)
    kk_q = set(query.get('kata_kunci', []))
    kk_d = set(kasus_db.get('fitur', {}).get('kata_kunci', []))
    if kk_q and kk_d:
        irisan = kk_q & kk_d
        gabung = kk_q | kk_d
        skor += bobot['kata_kunci'] * (len(irisan) / len(gabung))

    # 2. Kecocokan jenis perkara
    if query.get('jenis_perkara', '').lower() == kasus_db.get('kategori', '').lower():
        skor += bobot['jenis_perkara']

    # 3. Kecocokan pengadilan
    q_pengadilan = query.get('pengadilan', '').lower()
    db_pengadilan = kasus_db.get('pengadilan', '').lower()
    if q_pengadilan and q_pengadilan in db_pengadilan:
        skor += bobot['pengadilan']

    # 4. Kesamaan teks (Jaccard kata)
    kata_q = set(re.findall(r'\b[a-zA-Z]{4,}\b', query.get('deskripsi', '').lower()))
    kata_d = set(re.findall(r'\b[a-zA-Z]{4,}\b', kasus_db.get('teks_ringkas', '')[:500].lower()))
    if kata_q and kata_d:
        irisan_t = kata_q & kata_d
        gabung_t = kata_q | kata_d
        skor += bobot['teks'] * (len(irisan_t) / len(gabung_t))

    return round(skor, 4)


def cari_kasus_serupa(deskripsi, jenis_perkara='', pengadilan='', top_k=5):
    if not case_base:
        print('Case base kosong.')
        return []

    query = {
        'deskripsi'    : deskripsi,
        'kata_kunci'   : ekstrak_kata_kunci(deskripsi),
        'jenis_perkara': jenis_perkara,
        'pengadilan'   : pengadilan,
    }

    hasil = []
    for kasus in case_base:
        skor = hitung_similaritas(query, kasus)
        hasil.append({
            'skor'       : skor,
            'case_id'    : kasus.get('case_id', ''),
            'nomor'      : kasus.get('nomor', ''),
            'tanggal'    : kasus.get('tanggal', ''),
            'pengadilan' : kasus.get('pengadilan', ''),
            'para_pihak' : kasus.get('para_pihak', ''),
            'kategori'   : kasus.get('kategori', ''),
            'problem'    : kasus.get('problem', '')[:200],
            'solution'   : kasus.get('solution', '')[:200],
            'url'        : kasus.get('url', ''),
        })

    hasil.sort(key=lambda x: x['skor'], reverse=True)
    return hasil[:top_k]


print('Fungsi CBR siap.')
print('Total kasus dalam case base: ' + str(len(case_base)))

## Demo: Cari Kasus Serupa

In [ ]:
print('=' * 60)
print('  DEMO CBR: CARI KASUS SERUPA')
print('=' * 60)
print()
print('Contoh: sengketa tanah waris antara ahli waris atas sertifikat hak milik')
print()

DESKRIPSI   = input('Deskripsi kasus yang dicari: ').strip()
JENIS       = input('Filter jenis perkara [kosong=semua]: ').strip()
PENGADILAN  = input('Filter pengadilan [kosong=semua]: ').strip()
try:
    TOP_K = int(input('Tampilkan berapa hasil [default=5]: ').strip() or '5')
except ValueError:
    TOP_K = 5

if not DESKRIPSI:
    print('Query kosong, demo dilewati.')
else:
    hasil = cari_kasus_serupa(DESKRIPSI, JENIS, PENGADILAN, TOP_K)

    print()
    print('=' * 60)
    print('  HASIL PENCARIAN CBR')
    print('  Query: ' + DESKRIPSI[:70])
    print('=' * 60)

    if not hasil:
        print('Tidak ditemukan kasus serupa.')
    else:
        for i, k in enumerate(hasil, 1):
            print()
            print('  [' + str(i) + '] Skor Similaritas: ' + str(round(k['skor']*100, 1)) + '%')
            print('      Nomor     : ' + str(k['nomor']))
            print('      Tanggal   : ' + str(k['tanggal']))
            print('      Pengadilan: ' + str(k['pengadilan']))
            print('      Para Pihak: ' + str(k['para_pihak'])[:70])
            print('      Masalah   : ' + str(k['problem'])[:100] + '...')
            print('      Solusi    : ' + str(k['solution'])[:100] + '...')
            print('      URL       : ' + str(k['url'])[:70])

## Simpan Hasil Pencarian

In [ ]:
if 'hasil' in dir() and hasil:
    df_hasil = pd.DataFrame(hasil)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    path_hasil = os.path.join(CASE_BASE_DIR, 'hasil_cbr_' + ts + '.csv')
    df_hasil.to_csv(path_hasil, index=False, encoding='utf-8-sig')
    print('Hasil disimpan: ' + path_hasil)
    print()
    display(df_hasil[['skor', 'nomor', 'tanggal', 'pengadilan', 'problem', 'solution']])
else:
    print('Tidak ada hasil untuk disimpan.')

## Ringkasan Seluruh Pipeline

| Tahap | Notebook | Input | Output |
|-------|----------|-------|--------|
| 1 | `01_scraping_metadata.ipynb` | Pilihan pengguna | `data/raw/metadata/metadata_*.csv` |
| 2 | `02_unduh_pdf.ipynb` | Metadata CSV | `data/raw/pdf/*.pdf` |
| 3 | `03_ekstrak_teks.ipynb` | PDF | `data/processed/teks_ekstrak/*.txt` |
| 4 | `04_bangun_case_base.ipynb` | TXT | `data/processed/case_base/case_base_*.json` |
| 5 | `05_demo_cbr.ipynb` | JSON case base | Hasil pencarian |

Gunakan file `case_base_*.json` atau `case_base_*.csv` sebagai dataset utama untuk analisis CBR lebih lanjut.